## 🎯 Learning Objectives
* Understand the core principles and iterative nature of Self-RAG.
* Differentiate Self-RAG from traditional RAG and corrective RAG approaches.
* Implement a simplified Self-RAG workflow using LangGraph for iterative retrieval, critique, and generation.
* Analyze the performance trade-offs and identify suitable use cases for Self-RAG systems.


## Self-RAG: Retrieve, Critique, Generate, Regenerate

In the evolving landscape of Retrieval Augmented Generation (RAG), Self-RAG emerges as a powerful paradigm designed to significantly enhance the factual consistency and overall quality of generated responses. While traditional RAG systems retrieve documents and then generate an answer in a single pass, Self-RAG introduces an iterative, self-correcting mechanism, mimicking a human's critical thinking process.

### The Core Idea: Iterative Refinement

Imagine a student writing a research paper. They first gather information (retrieve), then draft a section. Before submitting, a diligent student (or a peer reviewer) would critically evaluate their draft: "Is this accurate? Is it well-supported by the sources? Are there any gaps or inconsistencies?" Based on this critique, they might go back to their sources, find more information, and revise their draft. Self-RAG operates on this very principle.

Instead of a single `retrieve -> generate` step, Self-RAG introduces a `critique` phase. After an initial generation, a specialized Large Language Model (LLM) acts as a critic, evaluating the generated response against the retrieved documents and the original query. If the critique identifies issues (e.g., hallucination, lack of support, incompleteness), the system can then `regenerate` the answer, potentially with new or refined retrieval, until a satisfactory response is produced.

### Why Self-RAG?

Traditional RAG, while effective, can still suffer from:
*   **Hallucinations**: Generating plausible but factually incorrect information.
*   **Lack of grounding**: Producing answers not fully supported by the retrieved context.
*   **Incompleteness**: Missing crucial details even when relevant information is available.

Self-RAG directly addresses these by:
*   **Enhancing Factual Consistency**: The critique step explicitly checks for factual accuracy and alignment with sources.
*   **Reducing Hallucinations**: By identifying unsupported claims, the system can correct itself.
*   **Improving Response Quality**: Iterative refinement leads to more comprehensive and coherent answers.
*   **Increased Trustworthiness**: Users can have higher confidence in the generated output.

### The Self-RAG Loop: A Step-by-Step Breakdown

1.  **Retrieve**: Given a user query, the system retrieves relevant documents from a knowledge base.
2.  **Generate (Initial)**: An LLM generates an initial answer based on the query and the retrieved documents.
3.  **Critique**: A separate LLM (or the same LLM with a different prompt) evaluates the generated answer. This critique often involves assessing:
    *   **Factual Accuracy**: Is the answer consistent with the retrieved documents?
    *   **Completeness**: Does the answer address all aspects of the query?
    *   **Coherence**: Is the answer well-structured and easy to understand?
    *   **Grounding**: Is every statement supported by the provided context?
4.  **Decide to Regenerate**: Based on the critique, a decision node determines if the answer is satisfactory or if further refinement is needed. This might involve setting a confidence threshold or checking for specific critique flags.
5.  **Regenerate (and potentially Re-retrieve)**: If regeneration is needed, the system might:
    *   Refine the original query for a new retrieval attempt.
    *   Generate a new answer using the same or refined retrieved documents, incorporating feedback from the critique.
    *   Iterate through the `critique` and `regenerate` steps until the answer meets the quality criteria or a maximum number of iterations is reached.

This iterative process, orchestrated beautifully by frameworks like LangGraph, allows for dynamic, self-improving RAG systems that push the boundaries of what's possible in accurate and reliable information retrieval.


In [ ]:
import os
from typing import List, Dict, Any
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import SentenceTransformerEmbeddings
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# Ensure you have your OpenAI API key set as an environment variable
# os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"

# --- 1. Setup Environment and Components ---

# Initialize LLMs (using gpt-4o-mini for cost-effectiveness and capability in 2026)
# For local development, consider models like Llama 3 or Mixtral via Ollama/vLLM
generator_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
critic_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

# Initialize Embedding Model
# Using a robust open-source embedding model for general purpose embeddings
embeddings = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

# Create a dummy vector store for demonstration
# In a real application, this would be populated from a large corpus
docs = [
    Document(page_content="LangGraph is a library for building stateful, multi-actor applications with LLMs.", metadata={"source": "langgraph_docs"}),
    Document(page_content="It extends LangChain by adding the ability to create cyclic graphs, enabling more complex agentic behaviors.", metadata={"source": "langgraph_docs"}),
    Document(page_content="Self-RAG is an advanced RAG technique that involves iterative critique and regeneration.", metadata={"source": "rag_paper"}),
    Document(page_content="The core idea of Self-RAG is to use an LLM to critique its own generated answers.", metadata={"source": "rag_paper"}),
    Document(page_content="Corrective RAG focuses on identifying and correcting retrieval failures.", metadata={"source": "rag_types"}),
    Document(page_content="Agentic RAG systems often combine multiple tools and decision-making steps.", metadata={"source": "agentic_ai"})
]
vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(k=2)

# --- 2. Define LangGraph State --- 

# The state will hold all information passed between nodes
class GraphState(Dict): 
    """Represents the state of our graph."""
    question: str
    documents: List[Document]
    generation: str
    critique: str
    iterations: int = 0
    max_iterations: int = 3

# --- 3. Define Graph Nodes ---

def retrieve(state: GraphState) -> GraphState:
    """Retrieve documents based on the question."""
    print("---RETRIEVE---")
    question = state["question"]
    documents = retriever.invoke(question)
    return {"documents": documents, "question": question}

def generate(state: GraphState) -> GraphState:
    """Generate an answer based on retrieved documents and question."""
    print("---GENERATE---")
    question = state["question"]
    documents = state["documents"]

    # Prompt for generation
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert AI assistant. Answer the user's question based *only* on the provided context. If you cannot find the answer, state that you don't know."),
        ("human", "Context: {context}\nQuestion: {question}")
    ])
    
    # Format context for the prompt
    context = "\n\n".join([doc.page_content for doc in documents])
    
    chain = prompt | generator_llm | StrOutputParser()
    generation = chain.invoke({"context": context, "question": question})
    return {"generation": generation, "question": question, "documents": documents}

def critique(state: GraphState) -> GraphState:
    """Critique the generated answer for factual accuracy and completeness."""
    print("---CRITIQUE---")
    question = state["question"]
    documents = state["documents"]
    generation = state["generation"]
    iterations = state["iterations"]

    # Prompt for critique
    critique_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a critical AI assistant. Your task is to evaluate a generated answer based on the provided context and question. \n" 
                   "Provide a concise critique. If the answer is factually incorrect, incomplete, or not fully supported by the context, explain why. \n" 
                   "If the answer is good, simply state 'GOOD'."),
        ("human", "Question: {question}\nContext: {context}\nGenerated Answer: {generation}\nCritique:")
    ])

    context = "\n\n".join([doc.page_content for doc in documents])
    chain = critique_prompt | critic_llm | StrOutputParser()
    critique_result = chain.invoke({"question": question, "context": context, "generation": generation})
    
    return {"critique": critique_result, "iterations": iterations + 1}

# --- 4. Define Conditional Edges ---

def decide_to_regenerate(state: GraphState) -> str:
    """Decide whether to regenerate the answer based on the critique."""
    print("---DECIDE TO REGENERATE---")
    critique_result = state["critique"]
    iterations = state["iterations"]
    max_iterations = state["max_iterations"]

    # Simple heuristic: if critique is not 'GOOD' and we haven't exceeded max iterations
    if "GOOD" not in critique_result.upper() and iterations < max_iterations:
        print(f"Critique: {critique_result}. Regenerating... (Iteration {iterations}/{max_iterations})")
        return "regenerate"
    else:
        print(f"Critique: {critique_result}. Final answer. (Iteration {iterations}/{max_iterations})")
        return "end"

# --- 5. Build the LangGraph Workflow ---

workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)
workflow.add_node("critique", critique)

# Set entry point
workflow.set_entry_point("retrieve")

# Add edges
workflow.add_edge("retrieve", "generate")
workflow.add_edge("generate", "critique")

# Add conditional edge for regeneration
workflow.add_conditional_edge(
    "critique",
    decide_to_regenerate,
    {
        "regenerate": "generate", # Loop back to generate (could also go to retrieve for new docs)
        "end": END
    }
)

# Compile the graph
app = workflow.compile()

# --- 6. Run the Self-RAG System ---

# Example Query 1: Simple, direct answer
print("\n--- Running Self-RAG for Query 1 ---")
query1 = "What is LangGraph?"
inputs1 = {"question": query1, "documents": [], "generation": "", "critique": "", "iterations": 0, "max_iterations": 2}
for s in app.stream(inputs1):
    print(s)
    print("----")

# Example Query 2: Requires more specific grounding, potentially leading to critique
print("\n--- Running Self-RAG for Query 2 ---")
query2 = "How does Self-RAG improve upon traditional RAG?"
inputs2 = {"question": query2, "documents": [], "generation": "", "critique": "", "iterations": 0, "max_iterations": 3}
for s in app.stream(inputs2):
    print(s)
    print("----")

# Example Query 3: A query that might be hard to fully answer with limited context
print("\n--- Running Self-RAG for Query 3 ---")
query3 = "What are the main differences between Self-RAG and Corrective RAG?"
inputs3 = {"question": query3, "documents": [], "generation": "", "critique": "", "iterations": 0, "max_iterations": 3}
for s in app.stream(inputs3):
    print(s)
    print("----")


### Interpreting the Code Output and Performance Trade-offs

The code above demonstrates a simplified Self-RAG loop using LangGraph. When you run it, you'll observe the system progressing through distinct stages:

1.  **`---RETRIEVE---`**: The system fetches relevant documents based on the initial query.
2.  **`---GENERATE---`**: An initial answer is formulated using the retrieved context.
3.  **`---CRITIQUE---`**: The critic LLM evaluates the generated answer. You'll see its output, which might be "GOOD" or a detailed explanation of issues.
4.  **`---DECIDE TO REGENERATE---`**: Based on the critique, the system decides whether to loop back to `generate` (or potentially `retrieve` if the critique suggests new information is needed) or `end` the process.

For `Query 1` ("What is LangGraph?"), the system might quickly arrive at a "GOOD" critique and terminate, as the information is straightforward and directly available. For `Query 2` ("How does Self-RAG improve upon traditional RAG?"), the initial generation might be incomplete or lack specific details, leading the critic to flag it. This triggers a regeneration, where the generator LLM attempts to produce a better answer, potentially leading to a more satisfactory final output.

`Query 3` ("What are the main differences between Self-RAG and Corrective RAG?") is designed to be challenging with our limited dummy context. The system might struggle to provide a comprehensive answer, and the critic might repeatedly flag it as incomplete or not fully supported. After a few iterations (controlled by `max_iterations`), the system will eventually terminate, even if the answer isn't perfect, demonstrating the iteration limit.

#### Performance Trade-offs

Self-RAG, while powerful, comes with inherent trade-offs:

*   **Increased Latency**: Each iteration involves multiple LLM calls (generation, critique) and potentially retrieval. This significantly increases the time taken to produce a final answer compared to a single-pass RAG system. For real-time applications, this can be a critical bottleneck.
*   **Higher Computational Cost**: More LLM calls directly translate to higher API costs (for commercial models) or increased computational resources (for self-hosted models). Careful management of `max_iterations` and efficient prompt engineering for the critic are crucial.
*   **Complexity in Prompt Engineering**: Designing effective prompts for both the generator and, especially, the critic is vital. A poorly designed critic prompt might be too lenient (missing errors) or too strict (leading to endless regeneration loops).
*   **State Management**: Managing the state across iterations (e.g., tracking previous generations, critiques, and retrieved documents) adds complexity to the system design, though frameworks like LangGraph simplify this considerably.

#### Typical Use Cases

Despite the trade-offs, Self-RAG is invaluable for scenarios demanding high accuracy and reliability:

*   **High-Stakes Question Answering**: Applications in legal, medical, or financial domains where factual errors can have severe consequences.
*   **Complex Research Tasks**: Assisting researchers in synthesizing information from vast document sets, ensuring all claims are well-supported.
*   **Content Generation with Factual Constraints**: Creating articles, reports, or educational materials where factual accuracy is paramount.
*   **Automated Fact-Checking**: Systems designed to verify information against a trusted knowledge base.
*   **Customer Support for Critical Information**: Providing accurate and consistent answers to customer queries about product specifications, policies, or troubleshooting steps.

By carefully balancing the need for accuracy with the operational costs, Self-RAG can be deployed to build truly robust and trustworthy agentic RAG systems.


### Resources

*   **LangGraph Documentation**: The official guide for building stateful, multi-actor applications with LLMs. [https://langchain-ai.github.io/langgraph/](https://langchain-ai.github.io/langgraph/)
*   **LangChain RAG Documentation**: Comprehensive resources on various RAG techniques and implementations within LangChain. [https://python.langchain.com/docs/use_cases/question_answering/](https://python.langchain.com/docs/use_cases/question_answering/)
*   **Original Self-RAG Paper (or a good summary)**: While the original paper is a good read, often a well-explained blog post or summary can provide quicker insights. Search for "Self-RAG: Learning to Retrieve, Generate, and Critique through Self-Reflection" by Akari Asai et al. (2023).
*   **OpenAI API Documentation**: For details on using `gpt-4o-mini` and other models. [https://platform.openai.com/docs/](https://platform.openai.com/docs/)
*   **Hugging Face Transformers & Sentence Transformers**: For exploring and utilizing various embedding models. [https://huggingface.co/docs/transformers/index](https://huggingface.co/docs/transformers/index) and [https://www.sbert.net/](https://www.sbert.net/)
*   **FAISS Documentation**: For efficient similarity search and dense vector indexing. [https://faiss.ai/](https://faiss.ai/)
